In [ ]:
"""
Script 01 — Wasserstein W1 (moyenne des SNRs)
================================================

Tâche unique : calculer la distance de Wasserstein W1 entre les distributions
réelles et simulées par modèle pour chaque SNR, puis moyenner ces distances
sur les SNRs, en 5-fold cross-validation.

Source des données : Distributions_Active_late.csv
    - 20 participants, 5 folds, 3 modèles comparés

Sorties :
    - Affichage console : tableau récapitulatif + test Wilcoxon
    - Figure : boxplot du W1 moyen par modèle (population + individus)
"""

import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance, wilcoxon
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Headless mode
import os

# ─────────────────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────────────────

NOTEBOOK_DIR = '/home/thardy/elefanto/ConsciousnessTeam_Data/SOUNDMODEL/Data_SoundGOOD/LEAD_ExperimentalFolder/SOUNDMODEL_clean'
DATA_PATH = os.path.join(NOTEBOOK_DIR, '../Distributions_Active_late.csv')
OUTPUT_DIR = NOTEBOOK_DIR

MODELS = {
    'linear': 'Linear',
    'nonlinear': 'NonLinear1',
    'gainmodul': 'GainModulation',
}
N_PARTICIPANTS = 20
N_FOLDS = 5


# ─────────────────────────────────────────────────────────────────────────────
# Chargement des données
# ─────────────────────────────────────────────────────────────────────────────

print("Chargement du CSV...")
df = pd.read_csv(DATA_PATH)
data_cols = [c for c in df.columns if c.startswith('idx_')]
if 'snr' not in df.columns:
    raise ValueError("La colonne 'snr' est absente du CSV.")
if len(data_cols) == 0:
    raise ValueError("Aucune colonne 'idx_' trouvée dans le CSV.")
if df['snr'].isna().any():
    raise ValueError("La colonne 'snr' contient des NaN. Nettoyez les données avant calcul.")
snr_values = sorted(df['snr'].unique().tolist())
print(f"  Shape: {df.shape} | Modèles disponibles: {sorted(df['model'].unique())}")
print(f"  SNRs détectés: {snr_values}")

# ─────────────────────────────────────────────────────────────────────────────
# Préparation des distributions par SNR
# ─────────────────────────────────────────────────────────────────────────────
print("\nPréparation des distributions par SNR...")
per_snr_distributions = {}
for participant in sorted(df['participant'].unique()):
    for fold in sorted(df['fold'].unique()):
        for model in sorted(df['model'].unique()):
            for snr in snr_values:
                subset = df[
                    (df['participant'] == participant) &
                    (df['fold'] == fold) &
                    (df['model'] == model) &
                    (df['snr'] == snr)
                ]
                if len(subset) > 0:
                    key = (participant, fold, model, snr)
                    all_vals = []
                    for _, row in subset.iterrows():
                        for col in data_cols:
                            val = pd.to_numeric(row[col], errors='coerce')
                            if not pd.isna(val):
                                all_vals.append(val)
                    if len(all_vals) == 0:
                        raise ValueError(
                            f"Distribution vide après suppression des NaN pour "
                            f"participant={participant}, fold={fold}, model={model}, snr={snr}"
                        )
                    per_snr_distributions[key] = np.asarray(all_vals, dtype=float)
print(f"  Préparation complétée : {len(per_snr_distributions)} distributions créées")

# Vérification stricte de complétude pour les combinaisons attendues
expected_models = ['real'] + list(MODELS.keys())
missing_keys = []
for p in range(N_PARTICIPANTS):
    for f in range(N_FOLDS):
        for m in expected_models:
            for snr in snr_values:
                k = (p, f, m, snr)
                if k not in per_snr_distributions:
                    missing_keys.append(k)

if missing_keys:
    preview = ', '.join([str(k) for k in missing_keys[:5]])
    raise ValueError(
        f"Données manquantes pour {len(missing_keys)} combinaisons (p, f, model, snr). "
        f"Exemples: {preview}"
    )


def get_distribution(participant, fold, model, snr):
    """Retourne les valeurs d'une distribution pour un SNR donné (sans NaN)."""
    key = (participant, fold, model, snr)
    if key not in per_snr_distributions:
        raise ValueError(
            f"Aucune donnée pour participant={participant}, fold={fold}, model={model}, snr={snr}"
        )
    return per_snr_distributions[key]


# ─────────────────────────────────────────────────────────────────────────────
# Calcul W1 par participant et par fold (moyenne des W1 par SNR)
# ─────────────────────────────────────────────────────────────────────────────

print("\nCalcul W1 pour chaque participant / fold / modèle (moyenne sur SNRs)...")
results = {label: np.zeros((N_PARTICIPANTS, N_FOLDS)) for label in MODELS.values()}

for p in range(N_PARTICIPANTS):
    for f in range(N_FOLDS):
        for model_key, model_label in MODELS.items():
            w1_per_snr = []
            for snr in snr_values:
                real = get_distribution(p, f, 'real', snr)
                sim = get_distribution(p, f, model_key, snr)
                w1_per_snr.append(wasserstein_distance(real, sim))
            results[model_label][p, f] = float(np.mean(w1_per_snr))

# ─────────────────────────────────────────────────────────────────────────────
# Agrégation : score par participant = moyenne des 5 folds
# ─────────────────────────────────────────────────────────────────────────────

scores = {label: results[label].mean(axis=1) for label in MODELS.values()}

print("\n=== Résultats W1 moyen par SNR (moyenne sur 5 folds) ===")
print(f"{'Modèle':<20} {'Mean':>8} {'Median':>8} {'Std':>8}")
print("-" * 46)
for label in MODELS.values():
    s = scores[label]
    print(f"{label:<20} {s.mean():>8.4f} {np.median(s):>8.4f} {s.std():>8.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# Tests statistiques : Wilcoxon signé (comparaisons par paires)
# ─────────────────────────────────────────────────────────────────────────────

print("\n=== Tests de Wilcoxon (comparaisons par paires) ===")
model_labels = list(MODELS.values())
for i in range(len(model_labels)):
    for j in range(i + 1, len(model_labels)):
        a, b = model_labels[i], model_labels[j]
        stat, p = wilcoxon(scores[a], scores[b])
        better = a if scores[a].mean() < scores[b].mean() else b
        print(f"  {a} vs {b}: p={p:.4f} - gagne: {better}")

# ─────────────────────────────────────────────────────────────────────────────
# Modèle gagnant par participant
# ─────────────────────────────────────────────────────────────────────────────

print("\n=== Modèle gagnant par participant (W1 le plus bas) ===")
winner_counts = {label: 0 for label in MODELS.values()}
for p in range(N_PARTICIPANTS):
    winner = min(MODELS.values(), key=lambda m: scores[m][p])
    winner_counts[winner] += 1
    print(f"  P{p:02d}: {winner:<20} (W1={scores[winner][p]:.4f})")

print(f"\nRécapitulatif : {winner_counts}")

# ─────────────────────────────────────────────────────────────────────────────
# Figure
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 5))
x_labels = list(MODELS.values())
data_to_plot = [scores[label] for label in x_labels]

bp = ax.boxplot(data_to_plot, labels=x_labels, patch_artist=True, notch=False, widths=0.5)
colors = ['#4C72B0', '#DD8452', '#55A868']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Overlay individual participants
for i, label in enumerate(x_labels):
    jitter = np.random.uniform(-0.1, 0.1, N_PARTICIPANTS)
    ax.scatter(np.full(N_PARTICIPANTS, i + 1) + jitter, scores[label],
               color='k', alpha=0.5, s=20, zorder=5)

ax.set_ylabel('W1 moyen (moyenné sur SNRs)', fontsize=12)
ax.set_title('Wasserstein W1 - Moyenne par SNR', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, 'ModelComparison_Wasserstein_result.png')
plt.savefig(fig_path, dpi=150)
plt.close()

Chargement du CSV...
  Shape: (2400, 5004) | Modèles disponibles: ['gainmodul', 'linear', 'nonlinear', 'real']
  SNRs détectés: [0, 1, 2, 3, 4, 5]

Préparation des distributions par SNR...
  Préparation complétée : 2400 distributions créées

Calcul W1 pour chaque participant / fold / modèle (moyenne sur SNRs)...

=== Résultats W1 moyen par SNR (moyenne sur 5 folds) ===
Modèle                   Mean   Median      Std
----------------------------------------------
Linear                 0.2496   0.2445   0.0359
NonLinear1             0.2300   0.2287   0.0274
GainModulation         0.2129   0.2131   0.0248

=== Tests de Wilcoxon (comparaisons par paires) ===
  Linear vs NonLinear1: p=0.0000 - gagne: NonLinear1
  Linear vs GainModulation: p=0.0000 - gagne: GainModulation
  NonLinear1 vs GainModulation: p=0.0000 - gagne: GainModulation

=== Modèle gagnant par participant (W1 le plus bas) ===
  P00: GainModulation       (W1=0.2630)
  P01: GainModulation       (W1=0.2331)
  P02: GainModulatio

/tmp/ipykernel_1054775/562186282.py:200: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_to_plot, labels=x_labels, patch_artist=True, notch=False, widths=0.5)


In [7]:
"""
Script 02 — Wasserstein W1 (moyenne des SNRs)
================================================

Tâche unique : calculer la distance de Wasserstein W1 entre les distributions
réelles et simulées par modèle pour chaque SNR, puis moyenner ces distances
sur les SNRs disponibles, en 5-fold cross-validation.

Source des données : Distributions_Passive_late.csv
    - 20 participants, 5 folds, 3 modèles comparés

Sorties :
    - Affichage console : tableau récapitulatif + test Wilcoxon
    - Figure : boxplot du W1 moyen par modèle (population + individus)
"""

import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance, wilcoxon
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Headless mode
import os

# ─────────────────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────────────────

NOTEBOOK_DIR = '/home/thardy/elefanto/ConsciousnessTeam_Data/SOUNDMODEL/Data_SoundGOOD/LEAD_ExperimentalFolder/SOUNDMODEL_clean'
DATA_PATH = os.path.join(NOTEBOOK_DIR, '../Distributions_Passive_late.csv')
OUTPUT_DIR = NOTEBOOK_DIR

MODELS = {
    'linear': 'Linear',
    'nonlinear': 'NonLinear1',
    'gainmodul': 'GainModulation',
}
N_PARTICIPANTS = 20
N_FOLDS = 5


# ─────────────────────────────────────────────────────────────────────────────
# Chargement des données
# ─────────────────────────────────────────────────────────────────────────────

print("Chargement du CSV...")
df = pd.read_csv(DATA_PATH)
data_cols = [c for c in df.columns if c.startswith('idx_')]
if 'snr' not in df.columns:
    raise ValueError("La colonne 'snr' est absente du CSV.")
if len(data_cols) == 0:
    raise ValueError("Aucune colonne 'idx_' trouvée dans le CSV.")
if df['snr'].isna().any():
    raise ValueError("La colonne 'snr' contient des NaN. Nettoyez les données avant calcul.")
snr_values = sorted(df['snr'].unique().tolist())
print(f"  Shape: {df.shape} | Modèles disponibles: {sorted(df['model'].unique())}")
print(f"  SNRs détectés: {snr_values}")

# ─────────────────────────────────────────────────────────────────────────────
# Préparation des distributions par SNR
# ─────────────────────────────────────────────────────────────────────────────
print("\nPréparation des distributions par SNR...")
per_snr_distributions = {}
for participant in sorted(df['participant'].unique()):
    for fold in sorted(df['fold'].unique()):
        for model in sorted(df['model'].unique()):
            for snr in snr_values:
                subset = df[
                    (df['participant'] == participant) &
                    (df['fold'] == fold) &
                    (df['model'] == model) &
                    (df['snr'] == snr)
                ]
                if len(subset) > 0:
                    key = (participant, fold, model, snr)
                    all_vals = []
                    for _, row in subset.iterrows():
                        for col in data_cols:
                            val = pd.to_numeric(row[col], errors='coerce')
                            if not pd.isna(val):
                                all_vals.append(val)
                    if len(all_vals) == 0:
                        raise ValueError(
                            f"Distribution vide après suppression des NaN pour "
                            f"participant={participant}, fold={fold}, model={model}, snr={snr}"
                        )
                    per_snr_distributions[key] = np.asarray(all_vals, dtype=float)
print(f"  Préparation complétée : {len(per_snr_distributions)} distributions créées")


def get_distribution(participant, fold, model, snr):
    """Retourne les valeurs d'une distribution pour un SNR donné (sans NaN)."""
    key = (participant, fold, model, snr)
    if key not in per_snr_distributions:
        raise ValueError(
            f"Aucune donnée pour participant={participant}, fold={fold}, model={model}, snr={snr}"
        )
    return per_snr_distributions[key]


# ─────────────────────────────────────────────────────────────────────────────
# Calcul W1 par participant et par fold (moyenne des W1 par SNR)
# ─────────────────────────────────────────────────────────────────────────────

print("\nCalcul W1 pour chaque participant / fold / modèle (moyenne sur SNRs disponibles)...")
results = {label: np.zeros((N_PARTICIPANTS, N_FOLDS)) for label in MODELS.values()}

for p in range(N_PARTICIPANTS):
    for f in range(N_FOLDS):
        for model_key, model_label in MODELS.items():
            w1_per_snr = []
            for snr in snr_values:
                real_key = (p, f, 'real', snr)
                sim_key = (p, f, model_key, snr)
                if real_key not in per_snr_distributions or sim_key not in per_snr_distributions:
                    continue
                real = per_snr_distributions[real_key]
                sim = per_snr_distributions[sim_key]
                w1_per_snr.append(wasserstein_distance(real, sim))
            if len(w1_per_snr) == 0:
                raise ValueError(
                    f"Aucun SNR commun disponible pour participant={p}, fold={f}, model={model_key}"
                )
            results[model_label][p, f] = float(np.mean(w1_per_snr))

# ─────────────────────────────────────────────────────────────────────────────
# Agrégation : score par participant = moyenne des 5 folds
# ─────────────────────────────────────────────────────────────────────────────

scores = {label: results[label].mean(axis=1) for label in MODELS.values()}

print("\n=== Résultats W1 moyen par SNR (moyenne sur 5 folds) ===")
print(f"{'Modèle':<20} {'Mean':>8} {'Median':>8} {'Std':>8}")
print("-" * 46)
for label in MODELS.values():
    s = scores[label]
    print(f"{label:<20} {s.mean():>8.4f} {np.median(s):>8.4f} {s.std():>8.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# Tests statistiques : Wilcoxon signé (comparaisons par paires)
# ─────────────────────────────────────────────────────────────────────────────

print("\n=== Tests de Wilcoxon (comparaisons par paires) ===")
model_labels = list(MODELS.values())
for i in range(len(model_labels)):
    for j in range(i + 1, len(model_labels)):
        a, b = model_labels[i], model_labels[j]
        stat, p = wilcoxon(scores[a], scores[b])
        better = a if scores[a].mean() < scores[b].mean() else b
        print(f"  {a} vs {b}: p={p:.4f} - gagne: {better}")

# ─────────────────────────────────────────────────────────────────────────────
# Modèle gagnant par participant
# ─────────────────────────────────────────────────────────────────────────────

print("\n=== Modèle gagnant par participant (W1 le plus bas) ===")
winner_counts = {label: 0 for label in MODELS.values()}
for p in range(N_PARTICIPANTS):
    winner = min(MODELS.values(), key=lambda m: scores[m][p])
    winner_counts[winner] += 1
    print(f"  P{p:02d}: {winner:<20} (W1={scores[winner][p]:.4f})")

print(f"\nRécapitulatif : {winner_counts}")

# ─────────────────────────────────────────────────────────────────────────────
# Figure
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 5))
x_labels = list(MODELS.values())
data_to_plot = [scores[label] for label in x_labels]

bp = ax.boxplot(data_to_plot, labels=x_labels, patch_artist=True, notch=False, widths=0.5)
colors = ['#4C72B0', '#DD8452', '#55A868']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Overlay individual participants
for i, label in enumerate(x_labels):
    jitter = np.random.uniform(-0.1, 0.1, N_PARTICIPANTS)
    ax.scatter(np.full(N_PARTICIPANTS, i + 1) + jitter, scores[label],
               color='k', alpha=0.5, s=20, zorder=5)

ax.set_ylabel('W1 moyen (moyenné sur SNRs)', fontsize=12)
ax.set_title('Wasserstein W1 - Moyenne par SNR', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, 'ModelComparison_Wasserstein_result.png')
plt.savefig(fig_path, dpi=150)
plt.close()

Chargement du CSV...
  Shape: (2600, 5004) | Modèles disponibles: ['gainmodul', 'linear', 'nonlinear', 'real']
  SNRs détectés: [0, 1, 2, 3, 4, 5, 6]

Préparation des distributions par SNR...
  Préparation complétée : 2600 distributions créées

Calcul W1 pour chaque participant / fold / modèle (moyenne sur SNRs disponibles)...

=== Résultats W1 moyen par SNR (moyenne sur 5 folds) ===
Modèle                   Mean   Median      Std
----------------------------------------------
Linear                 0.2306   0.2328   0.0360
NonLinear1             0.2132   0.2162   0.0292
GainModulation         0.2048   0.2006   0.0303

=== Tests de Wilcoxon (comparaisons par paires) ===
  Linear vs NonLinear1: p=0.0000 - gagne: NonLinear1
  Linear vs GainModulation: p=0.0000 - gagne: GainModulation
  NonLinear1 vs GainModulation: p=0.0007 - gagne: GainModulation

=== Modèle gagnant par participant (W1 le plus bas) ===
  P00: NonLinear1           (W1=0.2167)
  P01: GainModulation       (W1=0.2330)
  P02

/tmp/ipykernel_1054775/743503060.py:173: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_to_plot, labels=x_labels, patch_artist=True, notch=False, widths=0.5)


In [8]:
"""
Script 03 — Wasserstein W1 (moyenne des SNRs)
================================================

Tâche unique : calculer la distance de Wasserstein W1 entre les distributions
réelles et simulées par modèle pour chaque SNR, puis moyenner ces distances
sur les SNRs disponibles, en 5-fold cross-validation.

Source des données : Distributions_Active_early.csv
    - 20 participants, 5 folds, 3 modèles comparés

Sorties :
    - Affichage console : tableau récapitulatif + test Wilcoxon
    - Figure : boxplot du W1 moyen par modèle (population + individus)
"""

import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance, wilcoxon
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Headless mode
import os

# ─────────────────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────────────────

NOTEBOOK_DIR = '/home/thardy/elefanto/ConsciousnessTeam_Data/SOUNDMODEL/Data_SoundGOOD/LEAD_ExperimentalFolder/SOUNDMODEL_clean'
DATA_PATH = os.path.join(NOTEBOOK_DIR, '../Distributions_Active_early.csv')
OUTPUT_DIR = NOTEBOOK_DIR

MODELS = {
    'linear': 'Linear',
    'nonlinear': 'NonLinear1',
    'gainmodul': 'GainModulation',
}
N_PARTICIPANTS = 20
N_FOLDS = 5


# ─────────────────────────────────────────────────────────────────────────────
# Chargement des données
# ─────────────────────────────────────────────────────────────────────────────

print("Chargement du CSV...")
df = pd.read_csv(DATA_PATH)
data_cols = [c for c in df.columns if c.startswith('idx_')]
if 'snr' not in df.columns:
    raise ValueError("La colonne 'snr' est absente du CSV.")
if len(data_cols) == 0:
    raise ValueError("Aucune colonne 'idx_' trouvée dans le CSV.")
if df['snr'].isna().any():
    raise ValueError("La colonne 'snr' contient des NaN. Nettoyez les données avant calcul.")
snr_values = sorted(df['snr'].unique().tolist())
print(f"  Shape: {df.shape} | Modèles disponibles: {sorted(df['model'].unique())}")
print(f"  SNRs détectés: {snr_values}")

# ─────────────────────────────────────────────────────────────────────────────
# Préparation des distributions par SNR
# ─────────────────────────────────────────────────────────────────────────────
print("\nPréparation des distributions par SNR...")
per_snr_distributions = {}
for participant in sorted(df['participant'].unique()):
    for fold in sorted(df['fold'].unique()):
        for model in sorted(df['model'].unique()):
            for snr in snr_values:
                subset = df[
                    (df['participant'] == participant) &
                    (df['fold'] == fold) &
                    (df['model'] == model) &
                    (df['snr'] == snr)
                ]
                if len(subset) > 0:
                    key = (participant, fold, model, snr)
                    all_vals = []
                    for _, row in subset.iterrows():
                        for col in data_cols:
                            val = pd.to_numeric(row[col], errors='coerce')
                            if not pd.isna(val):
                                all_vals.append(val)
                    if len(all_vals) == 0:
                        raise ValueError(
                            f"Distribution vide après suppression des NaN pour "
                            f"participant={participant}, fold={fold}, model={model}, snr={snr}"
                        )
                    per_snr_distributions[key] = np.asarray(all_vals, dtype=float)
print(f"  Préparation complétée : {len(per_snr_distributions)} distributions créées")


def get_distribution(participant, fold, model, snr):
    """Retourne les valeurs d'une distribution pour un SNR donné (sans NaN)."""
    key = (participant, fold, model, snr)
    if key not in per_snr_distributions:
        raise ValueError(
            f"Aucune donnée pour participant={participant}, fold={fold}, model={model}, snr={snr}"
        )
    return per_snr_distributions[key]


# ─────────────────────────────────────────────────────────────────────────────
# Calcul W1 par participant et par fold (moyenne des W1 par SNR)
# ─────────────────────────────────────────────────────────────────────────────

print("\nCalcul W1 pour chaque participant / fold / modèle (moyenne sur SNRs disponibles)...")
results = {label: np.zeros((N_PARTICIPANTS, N_FOLDS)) for label in MODELS.values()}

for p in range(N_PARTICIPANTS):
    for f in range(N_FOLDS):
        for model_key, model_label in MODELS.items():
            w1_per_snr = []
            for snr in snr_values:
                real_key = (p, f, 'real', snr)
                sim_key = (p, f, model_key, snr)
                if real_key not in per_snr_distributions or sim_key not in per_snr_distributions:
                    continue
                real = per_snr_distributions[real_key]
                sim = per_snr_distributions[sim_key]
                w1_per_snr.append(wasserstein_distance(real, sim))
            if len(w1_per_snr) == 0:
                raise ValueError(
                    f"Aucun SNR commun disponible pour participant={p}, fold={f}, model={model_key}"
                )
            results[model_label][p, f] = float(np.mean(w1_per_snr))

# ─────────────────────────────────────────────────────────────────────────────
# Agrégation : score par participant = moyenne des 5 folds
# ─────────────────────────────────────────────────────────────────────────────

scores = {label: results[label].mean(axis=1) for label in MODELS.values()}

print("\n=== Résultats W1 moyen par SNR (moyenne sur 5 folds) ===")
print(f"{'Modèle':<20} {'Mean':>8} {'Median':>8} {'Std':>8}")
print("-" * 46)
for label in MODELS.values():
    s = scores[label]
    print(f"{label:<20} {s.mean():>8.4f} {np.median(s):>8.4f} {s.std():>8.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# Tests statistiques : Wilcoxon signé (comparaisons par paires)
# ─────────────────────────────────────────────────────────────────────────────

print("\n=== Tests de Wilcoxon (comparaisons par paires) ===")
model_labels = list(MODELS.values())
for i in range(len(model_labels)):
    for j in range(i + 1, len(model_labels)):
        a, b = model_labels[i], model_labels[j]
        stat, p = wilcoxon(scores[a], scores[b])
        better = a if scores[a].mean() < scores[b].mean() else b
        print(f"  {a} vs {b}: p={p:.4f} - gagne: {better}")

# ─────────────────────────────────────────────────────────────────────────────
# Modèle gagnant par participant
# ─────────────────────────────────────────────────────────────────────────────

print("\n=== Modèle gagnant par participant (W1 le plus bas) ===")
winner_counts = {label: 0 for label in MODELS.values()}
for p in range(N_PARTICIPANTS):
    winner = min(MODELS.values(), key=lambda m: scores[m][p])
    winner_counts[winner] += 1
    print(f"  P{p:02d}: {winner:<20} (W1={scores[winner][p]:.4f})")

print(f"\nRécapitulatif : {winner_counts}")

# ─────────────────────────────────────────────────────────────────────────────
# Figure
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 5))
x_labels = list(MODELS.values())
data_to_plot = [scores[label] for label in x_labels]

bp = ax.boxplot(data_to_plot, labels=x_labels, patch_artist=True, notch=False, widths=0.5)
colors = ['#4C72B0', '#DD8452', '#55A868']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Overlay individual participants
for i, label in enumerate(x_labels):
    jitter = np.random.uniform(-0.1, 0.1, N_PARTICIPANTS)
    ax.scatter(np.full(N_PARTICIPANTS, i + 1) + jitter, scores[label],
               color='k', alpha=0.5, s=20, zorder=5)

ax.set_ylabel('W1 moyen (moyenné sur SNRs)', fontsize=12)
ax.set_title('Wasserstein W1 - Moyenne par SNR', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, 'ModelComparison_Wasserstein_result.png')
# plt.savefig(fig_path, dpi=150)
plt.show()

Chargement du CSV...
  Shape: (2400, 5004) | Modèles disponibles: ['gainmodul', 'linear', 'nonlinear', 'real']
  SNRs détectés: [0, 1, 2, 3, 4, 5]

Préparation des distributions par SNR...
  Préparation complétée : 2400 distributions créées

Calcul W1 pour chaque participant / fold / modèle (moyenne sur SNRs disponibles)...

=== Résultats W1 moyen par SNR (moyenne sur 5 folds) ===
Modèle                   Mean   Median      Std
----------------------------------------------
Linear                 0.1976   0.1974   0.0317
NonLinear1             0.1977   0.2005   0.0302
GainModulation         0.1892   0.1903   0.0251

=== Tests de Wilcoxon (comparaisons par paires) ===
  Linear vs NonLinear1: p=0.4749 - gagne: Linear
  Linear vs GainModulation: p=0.0002 - gagne: GainModulation
  NonLinear1 vs GainModulation: p=0.0000 - gagne: GainModulation

=== Modèle gagnant par participant (W1 le plus bas) ===
  P00: GainModulation       (W1=0.2242)
  P01: GainModulation       (W1=0.1949)
  P02: GainM

/tmp/ipykernel_1054775/4178805740.py:173: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_to_plot, labels=x_labels, patch_artist=True, notch=False, widths=0.5)


In [9]:
"""
Script 04 — Wasserstein W1 (moyenne des SNRs)
================================================

Tâche unique : calculer la distance de Wasserstein W1 entre les distributions
réelles et simulées par modèle pour chaque SNR, puis moyenner ces distances
sur les SNRs disponibles, en 5-fold cross-validation.

Source des données : Distributions_Passive_early.csv
    - 20 participants, 5 folds, 3 modèles comparés

Sorties :
    - Affichage console : tableau récapitulatif + test Wilcoxon
    - Figure : boxplot du W1 moyen par modèle (population + individus)
"""

import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance, wilcoxon
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Headless mode
import os

# ─────────────────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────────────────

NOTEBOOK_DIR = '/home/thardy/elefanto/ConsciousnessTeam_Data/SOUNDMODEL/Data_SoundGOOD/LEAD_ExperimentalFolder/SOUNDMODEL_clean'
DATA_PATH = os.path.join(NOTEBOOK_DIR, '../Distributions_Passive_early.csv')
OUTPUT_DIR = NOTEBOOK_DIR

MODELS = {
    'linear': 'Linear',
    'nonlinear': 'NonLinear1',
    'gainmodul': 'GainModulation',
}
N_PARTICIPANTS = 20
N_FOLDS = 5


# ─────────────────────────────────────────────────────────────────────────────
# Chargement des données
# ─────────────────────────────────────────────────────────────────────────────

print("Chargement du CSV...")
df = pd.read_csv(DATA_PATH)
data_cols = [c for c in df.columns if c.startswith('idx_')]
if 'snr' not in df.columns:
    raise ValueError("La colonne 'snr' est absente du CSV.")
if len(data_cols) == 0:
    raise ValueError("Aucune colonne 'idx_' trouvée dans le CSV.")
if df['snr'].isna().any():
    raise ValueError("La colonne 'snr' contient des NaN. Nettoyez les données avant calcul.")
snr_values = sorted(df['snr'].unique().tolist())
print(f"  Shape: {df.shape} | Modèles disponibles: {sorted(df['model'].unique())}")
print(f"  SNRs détectés: {snr_values}")

# ─────────────────────────────────────────────────────────────────────────────
# Préparation des distributions par SNR
# ─────────────────────────────────────────────────────────────────────────────
print("\nPréparation des distributions par SNR...")
per_snr_distributions = {}
for participant in sorted(df['participant'].unique()):
    for fold in sorted(df['fold'].unique()):
        for model in sorted(df['model'].unique()):
            for snr in snr_values:
                subset = df[
                    (df['participant'] == participant) &
                    (df['fold'] == fold) &
                    (df['model'] == model) &
                    (df['snr'] == snr)
                ]
                if len(subset) > 0:
                    key = (participant, fold, model, snr)
                    all_vals = []
                    for _, row in subset.iterrows():
                        for col in data_cols:
                            val = pd.to_numeric(row[col], errors='coerce')
                            if not pd.isna(val):
                                all_vals.append(val)
                    if len(all_vals) == 0:
                        raise ValueError(
                            f"Distribution vide après suppression des NaN pour "
                            f"participant={participant}, fold={fold}, model={model}, snr={snr}"
                        )
                    per_snr_distributions[key] = np.asarray(all_vals, dtype=float)
print(f"  Préparation complétée : {len(per_snr_distributions)} distributions créées")


def get_distribution(participant, fold, model, snr):
    """Retourne les valeurs d'une distribution pour un SNR donné (sans NaN)."""
    key = (participant, fold, model, snr)
    if key not in per_snr_distributions:
        raise ValueError(
            f"Aucune donnée pour participant={participant}, fold={fold}, model={model}, snr={snr}"
)
    return per_snr_distributions[key]


# ─────────────────────────────────────────────────────────────────────────────
# Calcul W1 par participant et par fold (moyenne des W1 par SNR)
# ─────────────────────────────────────────────────────────────────────────────

print("\nCalcul W1 pour chaque participant / fold / modèle (moyenne sur SNRs disponibles)...")
results = {label: np.zeros((N_PARTICIPANTS, N_FOLDS)) for label in MODELS.values()}

for p in range(N_PARTICIPANTS):
    for f in range(N_FOLDS):
        for model_key, model_label in MODELS.items():
            w1_per_snr = []
            for snr in snr_values:
                real_key = (p, f, 'real', snr)
                sim_key = (p, f, model_key, snr)
                if real_key not in per_snr_distributions or sim_key not in per_snr_distributions:
                    continue
                real = per_snr_distributions[real_key]
                sim = per_snr_distributions[sim_key]
                w1_per_snr.append(wasserstein_distance(real, sim))
            if len(w1_per_snr) == 0:
                raise ValueError(
                    f"Aucun SNR commun disponible pour participant={p}, fold={f}, model={model_key}"
                )
            results[model_label][p, f] = float(np.mean(w1_per_snr))

# ─────────────────────────────────────────────────────────────────────────────
# Agrégation : score par participant = moyenne des 5 folds
# ─────────────────────────────────────────────────────────────────────────────

scores = {label: results[label].mean(axis=1) for label in MODELS.values()}

print("\n=== Résultats W1 moyen par SNR (moyenne sur 5 folds) ===")
print(f"{'Modèle':<20} {'Mean':>8} {'Median':>8} {'Std':>8}")
print("-" * 46)
for label in MODELS.values():
    s = scores[label]
    print(f"{label:<20} {s.mean():>8.4f} {np.median(s):>8.4f} {s.std():>8.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# Tests statistiques : Wilcoxon signé (comparaisons par paires)
# ─────────────────────────────────────────────────────────────────────────────

print("\n=== Tests de Wilcoxon (comparaisons par paires) ===")
model_labels = list(MODELS.values())
for i in range(len(model_labels)):
    for j in range(i + 1, len(model_labels)):
        a, b = model_labels[i], model_labels[j]
        stat, p = wilcoxon(scores[a], scores[b])
        better = a if scores[a].mean() < scores[b].mean() else b
        print(f"  {a} vs {b}: p={p:.4f} - gagne: {better}")

# ─────────────────────────────────────────────────────────────────────────────
# Modèle gagnant par participant
# ─────────────────────────────────────────────────────────────────────────────

print("\n=== Modèle gagnant par participant (W1 le plus bas) ===")
winner_counts = {label: 0 for label in MODELS.values()}
for p in range(N_PARTICIPANTS):
    winner = min(MODELS.values(), key=lambda m: scores[m][p])
    winner_counts[winner] += 1
    print(f"  P{p:02d}: {winner:<20} (W1={scores[winner][p]:.4f})")

print(f"\nRécapitulatif : {winner_counts}")

# ─────────────────────────────────────────────────────────────────────────────
# Figure
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 5))
x_labels = list(MODELS.values())
data_to_plot = [scores[label] for label in x_labels]

bp = ax.boxplot(data_to_plot, labels=x_labels, patch_artist=True, notch=False, widths=0.5)
colors = ['#4C72B0', '#DD8452', '#55A868']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Overlay individual participants
for i, label in enumerate(x_labels):
    jitter = np.random.uniform(-0.1, 0.1, N_PARTICIPANTS)
    ax.scatter(np.full(N_PARTICIPANTS, i + 1) + jitter, scores[label],
               color='k', alpha=0.5, s=20, zorder=5)

ax.set_ylabel('W1 moyen (moyenné sur SNRs)', fontsize=12)
ax.set_title('Wasserstein W1 - Moyenne par SNR', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, 'ModelComparison_Wasserstein_result.png')
# plt.savefig(fig_path, dpi=150)
plt.close()

Chargement du CSV...
  Shape: (2600, 5004) | Modèles disponibles: ['gainmodul', 'linear', 'nonlinear', 'real']
  SNRs détectés: [0, 1, 2, 3, 4, 5, 6]

Préparation des distributions par SNR...
  Préparation complétée : 2600 distributions créées

Calcul W1 pour chaque participant / fold / modèle (moyenne sur SNRs disponibles)...

=== Résultats W1 moyen par SNR (moyenne sur 5 folds) ===
Modèle                   Mean   Median      Std
----------------------------------------------
Linear                 0.1822   0.1845   0.0237
NonLinear1             0.1837   0.1864   0.0233
GainModulation         0.1781   0.1797   0.0236

=== Tests de Wilcoxon (comparaisons par paires) ===
  Linear vs NonLinear1: p=0.0583 - gagne: Linear
  Linear vs GainModulation: p=0.0064 - gagne: GainModulation
  NonLinear1 vs GainModulation: p=0.0000 - gagne: GainModulation

=== Modèle gagnant par participant (W1 le plus bas) ===
  P00: Linear               (W1=0.1977)
  P01: Linear               (W1=0.2118)
  P02: Ga

/tmp/ipykernel_1054775/441108792.py:173: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_to_plot, labels=x_labels, patch_artist=True, notch=False, widths=0.5)
